In [1]:
pip install pandas numpy faker

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker('en_IN')
np.random.seed(42)
random.seed(42)

# ---------------------------
# 1. CUSTOMERS (2000)
# ---------------------------
num_customers = 2000

customers = []
for i in range(1, num_customers + 1):
    customers.append({
        "customer_id": i,
        "name": fake.name(),
        "gender": random.choices(["Female", "Male"], weights=[0.75, 0.25])[0],
        "city": random.choice(["Chennai", "Bangalore", "Mumbai", "Delhi", "Hyderabad", "Pune"]),
        "signup_date": fake.date_between(start_date='-2y', end_date='today')
    })

customers_df = pd.DataFrame(customers)


# ---------------------------
# 2. PRODUCTS
# ---------------------------
brands = [
    "Lakme", "Maybelline", "L'Oreal", "Nykaa", "Mamaearth",
    "Minimalist", "Biotique", "WOW Skin Science", "Plum", "Sugar Cosmetics"
]

categories = {
    "Skincare": ["Cleanser", "Serum", "Moisturizer", "Sunscreen"],
    "Makeup": ["Lipstick", "Foundation", "Kajal", "Compact"],
    "Haircare": ["Shampoo", "Conditioner", "Hair Oil", "Hair Mask"]
}

products = []
product_id = 1

for category, items in categories.items():
    for item in items:
        for brand in brands:
            price = random.randint(200, 1800)
            cost = int(price * random.uniform(0.4, 0.7))
            
            products.append({
                "product_id": product_id,
                "product_name": f"{brand} {item}",
                "category": category,
                "brand": brand,
                "price": price,
                "cost_price": cost
            })
            product_id += 1

products_df = pd.DataFrame(products)


# ---------------------------
# 3. ORDERS
# ---------------------------
num_orders = 10000
orders = []

start_date = datetime(2023, 1, 1)

for i in range(1, num_orders + 1):
    
    if random.random() < 0.2:
        customer_id = random.randint(1, int(num_customers * 0.3))
    else:
        customer_id = random.randint(1, num_customers)
    
    order_date = start_date + timedelta(days=random.randint(0, 730))
    
    orders.append({
        "order_id": i,
        "customer_id": customer_id,
        "order_date": order_date,
        "order_status": random.choices(
            ["Delivered", "Cancelled", "Returned"],
            weights=[0.85, 0.1, 0.05]
        )[0]
    })

orders_df = pd.DataFrame(orders)


# ---------------------------
# 4. ORDER ITEMS
# ---------------------------
order_items = []

for _, order in orders_df.iterrows():
    
    num_items = random.randint(1, 4)
    
    if order['order_date'].month in [10, 11]:
        num_items += 1
    
    for _ in range(num_items):
        product = products_df.sample(1).iloc[0]
        
        discount = random.uniform(0.7, 0.95)
        quantity = random.randint(1, 3)
        selling_price = round(product['price'] * discount, 2)
        
        order_items.append({
            "order_id": order['order_id'],
            "product_id": product['product_id'],
            "quantity": quantity,
            "selling_price": selling_price,
            "discount_percent": round((1 - discount) * 100, 2),
            "total_price": round(quantity * selling_price, 2)
        })

order_items_df = pd.DataFrame(order_items)


# ---------------------------
# 5. ORDER VALUE
# ---------------------------
order_value_df = order_items_df.groupby("order_id")["total_price"].sum().reset_index()
order_value_df.rename(columns={"total_price": "order_value"}, inplace=True)

orders_df = orders_df.merge(order_value_df, on="order_id", how="left")


# ---------------------------
# 6. DELIVERY INFO
# ---------------------------
orders_df["delivery_date"] = orders_df["order_date"] + pd.to_timedelta(
    np.random.randint(2, 7, size=len(orders_df)), unit='D'
)

orders_df["delivery_time_days"] = (orders_df["delivery_date"] - orders_df["order_date"]).dt.days


# ---------------------------
# 7. PAYMENTS (REALISTIC FIX)
# ---------------------------
payments = []

methods = ["UPI", "Credit Card", "Debit Card", "Cash on Delivery"]

for i, row in orders_df.iterrows():

    if row['order_status'] == "Delivered":
        payment_status = "Completed"
        payment_value = row["order_value"]
    elif row['order_status'] == "Cancelled":
        payment_status = "Cancelled"
        payment_value = 0
    else:  # Returned
        payment_status = "Refunded"
        payment_value = 0

    payments.append({
        "payment_id": i + 1,
        "order_id": row['order_id'],
        "payment_method": random.choices(
            methods,
            weights=[0.5, 0.2, 0.2, 0.1]
        )[0],
        "payment_status": payment_status,
        "payment_value": round(payment_value, 2),
        "payment_date": row["order_date"] + timedelta(days=random.randint(0, 2))
    })

payments_df = pd.DataFrame(payments)


# ---------------------------
# 8. SAVE CSV
# ---------------------------
customers_df.to_csv("customers.csv", index=False)
products_df.to_csv("products.csv", index=False)
orders_df.to_csv("orders.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)
payments_df.to_csv("payments.csv", index=False)

print("✅ FINAL DATASET GENERATED (Industry-Level Ready)")

✅ FINAL DATASET GENERATED (Industry-Level Ready)
